In [1]:
!python -V

Python 3.10.15


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [32]:
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

2025/02/07 14:30:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/02/07 14:30:51 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

<Experiment: artifact_location='/Users/tobias/SEM7/dev/dtc-mlops/2_Experiment_Tracking/mlruns/1', creation_time=1738935052120, experiment_id='1', last_update_time=1738935052120, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [16]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [17]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [18]:
len(df_train), len(df_val)

(73908, 61921)

In [19]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [20]:
df_train.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration,PU_DO
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.5,...,0.00,0.0,None,0.3,6.80,2.0,1.0,0.00,3.933333,43_151
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.0,...,2.81,0.0,None,0.3,16.86,1.0,1.0,2.75,8.750000,166_239
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.0,...,1.00,0.0,None,0.3,8.30,1.0,1.0,0.00,5.966667,41_42
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.0,...,0.00,0.0,None,0.3,9.30,2.0,1.0,0.00,7.083333,168_75
7,2,2021-01-01 00:26:31,2021-01-01 00:28:50,N,1.0,75,75,6.0,0.45,3.5,...,0.96,0.0,None,0.3,5.76,1.0,1.0,0.00,2.316667,75_75


In [29]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
print(f'{train_dicts[:5]=}')
X_train = dv.fit_transform(train_dicts)
print(f"{X_train[:5]=}")

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

train_dicts[:5]=[{'PU_DO': '43_151', 'trip_distance': 1.01}, {'PU_DO': '166_239', 'trip_distance': 2.53}, {'PU_DO': '41_42', 'trip_distance': 1.12}, {'PU_DO': '168_75', 'trip_distance': 1.99}, {'PU_DO': '75_75', 'trip_distance': 0.45}]
X_train[:5]=<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10 stored elements and shape (5, 13221)>


In [30]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [33]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

# mean_squared_error(y_val, y_pred, squared=False)
sklearn.metrics.root_mean_squared_error(y_val, y_pred)

7.758715207732648

In [37]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [38]:
with mlflow.start_run():

    mlflow.set_tag("developer", "tobias")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    ls = Lasso(alpha)
    ls.fit(X_train, y_train)

    y_pred = ls.predict(X_val)
    rmse = sklearn.metrics.root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

# Tracking Hyperparams Study 

In [39]:
import xgboost as xgb

In [40]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [41]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [42]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")

        mlflow.log_params(params)

        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )

        y_pred = booster.predict(valid)

        rmse = sklearn.metrics.root_mean_squared_error(y_val, y_pred, )

        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [43]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/Users/tobias/SEM7/dev/dtc-mlops/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [15:10:42] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:7.85542                           
[1]	validation-rmse:6.86687                           
[2]	validation-rmse:6.64277                           
[3]	validation-rmse:6.57169                           
[4]	validation-rmse:6.53977                           
[5]	validation-rmse:6.52542                           
[6]	validation-rmse:6.50879                           
[7]	validation-rmse:6.50445                           
[8]	validation-rmse:6.49956                           
[9]	validation-rmse:6.49356                           
[10]	validation-rmse:6.49044                          
[11]	validation-rmse:6.48806                          
[12]	validation-rmse:6.48508                          
[13]	validation-rmse:6.48243                          
[14]	validation-rmse:6.48093                          
[15]	validation-rmse:6.47926                          
[16]	validation-rmse:6.47836                          
[17]	validation-rmse:6.47822                          
[18]	valid

/Users/tobias/SEM7/dev/dtc-mlops/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [15:10:53] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[3]	validation-rmse:10.50180                                                  
[4]	validation-rmse:10.15789                                                  
[5]	validation-rmse:9.84311                                                   
[6]	validation-rmse:9.55548                                                   
[7]	validation-rmse:9.29295                                                   
[8]	validation-rmse:9.05371                                                   
[9]	validation-rmse:8.83575                                                   
[10]	validation-rmse:8.63768                                                  
[11]	validation-rmse:8.45794                                                  
[12]	validation-rmse:8.29499                                                  
[13]	validation-rmse:8.14731                                                  
[14]	validation-rmse:8.01389                                                  
[15]	validation-rmse:7.89310                        

...

[2]	validation-rmse:10.97264                                                    
[3]	validation-rmse:10.61871                                                    
[4]	validation-rmse:10.29175                                                    
[5]	validation-rmse:9.98995                                                     
[6]	validation-rmse:9.71187                                                     
[7]	validation-rmse:9.45581                                                     
[8]	validation-rmse:9.22024                                                     
[9]	validation-rmse:9.00394                                                     
[10]	validation-rmse:8.80555                                                    
[11]	validation-rmse:8.62362                                                    
[12]	validation-rmse:8.45712                                                    
[13]	validation-rmse:8.30482                                                    
[14]	validation-rmse:8.16597

/Users/tobias/SEM7/dev/dtc-mlops/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [15:36:40] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[1]	validation-rmse:10.87599                                                    
[2]	validation-rmse:10.32210                                                    
[3]	validation-rmse:9.83497                                                     
[4]	validation-rmse:9.40783                                                     
[5]	validation-rmse:9.03428                                                     
[6]	validation-rmse:8.70969                                                     
[7]	validation-rmse:8.42713                                                     
[8]	validation-rmse:8.18141                                                     
[9]	validation-rmse:7.96913                                                     
[10]	validation-rmse:7.78574                                                    
[11]	validation-rmse:7.62749                                                    
[12]	validation-rmse:7.49195                                                    
[13]	validation-rmse:7.37432

/Users/tobias/SEM7/dev/dtc-mlops/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [15:37:21] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.49874                                                    
[1]	validation-rmse:9.26407                                                     
[2]	validation-rmse:8.38729                                                     
[3]	validation-rmse:7.77915                                                     
[4]	validation-rmse:7.36167                                                     
[5]	validation-rmse:7.07993                                                     
[6]	validation-rmse:6.88779                                                     
[7]	validation-rmse:6.76065                                                     
[8]	validation-rmse:6.66868                                                     
[9]	validation-rmse:6.60572                                                     
[10]	validation-rmse:6.56008                                                    
[11]	validation-rmse:6.52503                                                    
[12]	validation-rmse:6.49879

# Autologging the ReTraining of the best Model

In [44]:
mlflow.xgboost.autolog(disable=True)

In [48]:
import sklearn.metrics


with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = sklearn.metrics.root_mean_squared_error(y_val, y_pred, )
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/Users/tobias/SEM7/dev/dtc-mlops/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [15:55:31] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:11.44482
[1]	validation-rmse:10.77202
[2]	validation-rmse:10.18363
[3]	validation-rmse:9.67396
[4]	validation-rmse:9.23166
[5]	validation-rmse:8.84808
[6]	validation-rmse:8.51883
[7]	validation-rmse:8.23597
[8]	validation-rmse:7.99320
[9]	validation-rmse:7.78709
[10]	validation-rmse:7.61022
[11]	validation-rmse:7.45952
[12]	validation-rmse:7.33049
[13]	validation-rmse:7.22098
[14]	validation-rmse:7.12713
[15]	validation-rmse:7.04752
[16]	validation-rmse:6.98005
[17]	validation-rmse:6.92232
[18]	validation-rmse:6.87112
[19]	validation-rmse:6.82740
[20]	validation-rmse:6.78995
[21]	validation-rmse:6.75792
[22]	validation-rmse:6.72994
[23]	validation-rmse:6.70547
[24]	validation-rmse:6.68390
[25]	validation-rmse:6.66421
[26]	validation-rmse:6.64806
[27]	validation-rmse:6.63280
[28]	validation-rmse:6.61924
[29]	validation-rmse:6.60773
[30]	validation-rmse:6.59777
[31]	validation-rmse:6.58875
[32]	validation-rmse:6.58107
[33]	validation-rmse:6.57217
[34]	validation-rmse:

/Users/tobias/SEM7/dev/dtc-mlops/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [15:56:01] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/02/07 15:56:04 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/02/07 15:56:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [50]:
from mlflow.tracking import MlflowClient

# Initialize the MLflow client
client = MlflowClient()

# Specify the run ID
run_id = '59e84cdc3255460093b771f8a3691dec'  # Replace with your specific run ID

# Fetch the run data
run = client.get_run(run_id)

# Access and print the parameters
my_best_params = run.data.params
print(my_best_params)

{'learning_rate': '0.10832689852966451', 'max_depth': '66', 'min_child_weight': '1.1496860450328121', 'objective': 'reg:linear', 'reg_alpha': '0.10779820287439192', 'reg_lambda': '0.21235909119090116', 'seed': '42'}


In [51]:
mlflow.xgboost.autolog()
booster = xgb.train(
    params=my_best_params,
    dtrain=train,
    num_boost_round=1000,
    evals=[(valid, 'validation')],
    early_stopping_rounds=50
    )    


2025/02/07 15:59:58 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'd591feb9ec4f4c97a6beacee53d8dfd0', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow
/Users/tobias/SEM7/dev/dtc-mlops/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [16:00:01] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:11.33308
[1]	validation-rmse:10.57481
[2]	validation-rmse:9.92661
[3]	validation-rmse:9.37471
[4]	validation-rmse:8.90809
[5]	validation-rmse:8.51172
[6]	validation-rmse:8.17747
[7]	validation-rmse:7.89927
[8]	validation-rmse:7.66714
[9]	validation-rmse:7.47108
[10]	validation-rmse:7.30864
[11]	validation-rmse:7.17447
[12]	validation-rmse:7.06108
[13]	validation-rmse:6.96773
[14]	validation-rmse:6.88959
[15]	validation-rmse:6.82476
[16]	validation-rmse:6.76975
[17]	validation-rmse:6.72303
[18]	validation-rmse:6.68363
[19]	validation-rmse:6.65001
[20]	validation-rmse:6.62139
[21]	validation-rmse:6.59719
[22]	validation-rmse:6.57630
[23]	validation-rmse:6.55763
[24]	validation-rmse:6.54273
[25]	validation-rmse:6.52847
[26]	validation-rmse:6.51824
[27]	validation-rmse:6.50728
[28]	validation-rmse:6.49837
[29]	validation-rmse:6.48937
[30]	validation-rmse:6.48175
[31]	validation-rmse:6.47573
[32]	validation-rmse:6.47050
[33]	validation-rmse:6.46518
[34]	validation-rmse:6

2025/02/07 16:00:54 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2025/02/07 16:00:54 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/tobias/SEM7/dev/dtc-mlops/.venv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [16:00:54] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats."
2025/02/07 16:00:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/02/07 16:00:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [52]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        mlflow.log_metric("rmse", rmse)
        

2025/02/07 16:03:28 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'numpy.ndarray' object has no attribute 'toarray'
2025/02/07 16:07:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


TypeError: got an unexpected keyword argument 'squared'

# ML Client

In [60]:
#View all experiments
Experiments = client.search_experiments()
for exp in Experiments:
    print(f"Experiment#: {exp.experiment_id}, Experiment Name: {exp.name}")

Experiment#: 1, Experiment Name: nyc-taxi-experiment
Experiment#: 0, Experiment Name: Default


In [63]:
Experiments

[<Experiment: artifact_location='/Users/tobias/SEM7/dev/dtc-mlops/2_Experiment_Tracking/mlruns/1', creation_time=1738935052120, experiment_id='1', last_update_time=1738935052120, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='/Users/tobias/SEM7/dev/dtc-mlops/2_Experiment_Tracking/mlruns/0', creation_time=1738935052119, experiment_id='0', last_update_time=1738935052119, lifecycle_stage='active', name='Default', tags={}>]

In [76]:
from mlflow.entities import ViewType

#Select your chosen experiment
Exp_id = Experiments[0].experiment_id
Exp_name = Experiments[0].name
print(f"Experiment ID: {Exp_id}, Experiment Name: {Exp_name}")
#Get the runs for this experiment
runs = client.search_runs(
    experiment_ids = Exp_id,
    filter_string = "",
    run_view_type = ViewType.ACTIVE_ONLY,
    max_results = 10,
    order_by = ["metrics.rmse ASC"]
    )
for run in runs:
    print(f"run_id: {run.info.run_id}, metrics: {run.data.metrics}")

Experiment ID: 1, Experiment Name: nyc-taxi-experiment
run_id: 59e84cdc3255460093b771f8a3691dec, metrics: {'rmse': 6.309548285761134}
run_id: 429e08d091384d14a00919b5a169e44b, metrics: {'rmse': 6.310827415885453}
run_id: c37d12817615445da2d0cf8b23020ca6, metrics: {'rmse': 6.313172775604489}
run_id: 1ceba391a6ba4b568934251778beb1bb, metrics: {'rmse': 6.315440385758094}
run_id: bb656ef5e14747678e9cf8a68540be0a, metrics: {'rmse': 6.318445793399953}
run_id: 6257b3758dae44afada8a77dbb9e0ecc, metrics: {'rmse': 6.320881145686437}
run_id: 39670b91bcc34966a0367a385402fea2, metrics: {'rmse': 6.321179544355305}
run_id: fb51b813783e423d8341116e526cd6f2, metrics: {'rmse': 6.323258183080667}
run_id: 60b4a31493924febbd6c19bd31d09a59, metrics: {'rmse': 6.3260438341604335}
run_id: e3553c41c6fb4321ae48dc242a9ae80d, metrics: {'rmse': 6.328044749481757}


In [71]:
dict(runs[1].data.metrics)

{'validation-rmse': 6.3076373517450115,
 'stopped_iteration': 679.0,
 'best_iteration': 629.0}

In [69]:
runs[1].data.metrics['training_root_mean_squared_error']

2.402821301215763

In [70]:
for run in runs:
    print(f"run_id: {run.info.run_id}, rmse: {run.data.metrics['training_root_mean_squared_error']}")

run_id: b979e9fdee3b496984c76f7a2c344a33, rmse: 2.402821301215763


KeyError: 'training_root_mean_squared_error'